# Shapley values: background data changes the explanation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/statistics/shapley_values.ipynb)

A self-contained CPU notebook. All data are generated locally; no credentials or repository files are needed.

In [ ]:
%pip -q install numpy scipy scikit-learn matplotlib

## Experiment design

For each coalition, interventional replacement fixes the observed features and integrates missing features over their marginal Gaussian distribution. Conditional replacement instead uses the Gaussian distribution of missing features given the observed ones. The quadratic model lets us integrate exactly, so differences are not Monte Carlo noise.

All attributions add to prediction minus their own background expectation. The name interventional identifies a feature-replacement convention and does not establish a causal interpretation. A shifted reference population changes the baseline as well as the allocation.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import math


In [ ]:
def prediction(x):
    x=np.asarray(x);return x[...,0]+2*x[...,1]+x[...,0]*x[...,2]+.5*x[...,3]**2


In [ ]:
def coalition_values(x,mean,cov,conditional=False):
    """Exact Gaussian expectation of the quadratic prediction in every coalition."""
    values={};d=len(x)
    for mask in range(1<<d):
        fixed=np.array([j for j in range(d) if mask>>j&1],dtype=int)
        missing=np.array([j for j in range(d) if not mask>>j&1],dtype=int)
        m=mean.copy(); C=cov.copy()
        if len(fixed):
            if conditional and len(missing):
                gain=cov[np.ix_(missing,fixed)]@np.linalg.inv(cov[np.ix_(fixed,fixed)])
                m[missing]=mean[missing]+gain@(x[fixed]-mean[fixed])
                C[np.ix_(missing,missing)]-=gain@cov[np.ix_(fixed,missing)]
            m[fixed]=x[fixed];C[fixed,:]=0;C[:,fixed]=0
        values[mask]=float(prediction(m)+C[0,2]+.5*C[3,3])
    return values


In [ ]:
def exact_shapley(values,d=4):
    phi=np.zeros(d)
    for j in range(d):
        for mask in range(1<<d):
            if mask>>j&1:continue
            k=mask.bit_count();w=math.factorial(k)*math.factorial(d-k-1)/math.factorial(d)
            phi[j]+=w*(values[mask|1<<j]-values[mask])
    return phi


In [ ]:
def permutation_shapley(values,n=256,seed=0):
    rng=np.random.default_rng(seed); estimates=[]
    for _ in range(n):
        row=np.zeros(4);mask=0
        for j in rng.permutation(4):
            nxt=mask|1<<int(j);row[j]=values[nxt]-values[mask];mask=nxt
        estimates.append(row)
    return np.cumsum(estimates,axis=0)/np.arange(1,n+1)[:,None]


In [ ]:
def shapley_experiment():
    x=np.array([1.,1.,2.,1.]);mean=np.zeros(4);cov=np.eye(4);cov[0,1]=cov[1,0]=.8;cov[0,2]=cov[2,0]=.3
    assert np.linalg.eigvalsh(cov).min()>0
    scenarios={};all_values={}
    for label,m,conditional in [('Interventional',mean,False),('Conditional',mean,True),('Shifted background',np.array([.5,.5,0.,0.]),False)]:
        vals=coalition_values(x,m,cov,conditional);phi=exact_shapley(vals)
        assert abs(phi.sum()-(vals[15]-vals[0]))<1e-12
        scenarios[label]={'baseline':vals[0],'prediction':vals[15],'phi':phi.tolist()};all_values[label]=vals
    # Shapley symmetry and dummy checks for an additive game.
    test={mask:sum([1.,1.,0.,2.][j] for j in range(4) if mask>>j&1) for mask in range(16)}
    assert np.allclose(exact_shapley(test),[1,1,0,2])
    return scenarios,all_values,permutation_shapley(all_values['Interventional'])


In [ ]:
scenarios, values, estimates = shapley_experiment()
for name, result in scenarios.items():
    print(name, result)
fig, ax = plt.subplots(figsize=(8,4))
for i,(name,result) in enumerate(scenarios.items()):
    ax.bar(np.arange(4)+(i-1)*.24,result["phi"],.24,label=name)
ax.set_xticks(range(4),[f"Feature {i}" for i in range(4)])
ax.set_ylabel("Contribution"); ax.legend(); plt.show()
print("Permutation estimate",estimates[-1])


## Exercises

1. Set the correlations to zero and compare conditional with interventional values.
2. Set feature 3 to zero and explain the attribution of its squared term.
3. Add an interaction between features 1 and 3. Update the expectation formula before computing explanations.
4. Repeat permutation estimates over 30 random seeds and plot RMSE against permutation count.

## Tested environment
Executed on Python 3.12 with NumPy 2.5.3, SciPy 1.18.1, scikit-learn 1.9.1 and Matplotlib 3.11.2. Ranking additionally used XGBoost 3.4.1. Package versions and floating-point libraries can affect timings and fitted results.
